In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch
from tqdm.notebook import tqdm
import optuna

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

Path("results").mkdir(exist_ok=True)

print(f"PyTorch {torch.__version__} | GPU: {torch.cuda.is_available()}")

In [ ]:
from src.data.data_loader import get_dataset
from src.embeddings.graph_embeddings import compute_embeddings
from src.evaluation.community import (
    run_louvain, run_leiden, run_kmeans_emb, run_dmon, evaluate_communities, run_hdbscan
)

In [ ]:
datasets_to_load = ['youtube']

graphs = {}
for name in datasets_to_load:
    ds = get_dataset(name, verbose=True)
    graphs[name] = ds

In [ ]:
import sys, random, warnings
from pathlib import Path
import pandas as pd, numpy as np, networkx as nx
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.data_loader import get_dataset
from src.embeddings.graph_embeddings import compute_embeddings
from src.evaluation.community import *

yt = get_dataset('youtube', verbose=False)
community_sizes = [100, 200, 400]
emb_methods = ['node2vec', 'random_walk', 'grace', 'dgi']
results = []

def build_youtube_subgraph(dataset, n_communities=500, seed=None):
    communities = dataset['all_communities']
    if seed is not None:
        random.seed(seed)
        selected = random.sample(communities, min(n_communities, len(communities)))
    else:
        selected = communities[:n_communities]
    nodes_set = set().union(*selected)
    G_sub = dataset['graph'].subgraph(nodes_set).copy()
    labels = {n: i for i, comm in enumerate(selected) for n in comm if n in G_sub}
    return G_sub, labels

for k in tqdm(community_sizes, desc='Размеры подграфов'):
    G_sub, true_labels = build_youtube_subgraph(yt, n_communities=k, seed=42)
    n_clusters = len(set(true_labels.values()))

    # Загружаем/вычисляем все эмбеддинги
    emb_dict = {}
    for emb_name in emb_methods:
        emb_dict[emb_name] = compute_embeddings(
            G_sub, method=emb_name, dimensions=64,
            name=f"youtube_{k}_{emb_name}", seed=42
        )

    # Методы, не требующие эмбеддингов
    for method_func, method_name in [
        (run_louvain, 'Louvain'),
        (lambda: run_leiden(G_sub, seed=42, resolution=0.01), 'Leiden')
    ]:
        part = method_func()
        met = evaluate_communities(true_labels, part)
        results.append({
            'dataset': f'youtube_{k}', 'method': method_name,
            'nmi': met['nmi'], 'ari': met['ari'],
            'n_communities_found': len(set(part.values()))
        })

    # Методы с эмбеддингами: KMeans и HDBSCAN
    for emb_name, emb in emb_dict.items():
        # KMeans
        part_km = run_kmeans_emb(emb, n_clusters=n_clusters, random_state=42)
        met_km = evaluate_communities(true_labels, part_km)
        results.append({
            'dataset': f'youtube_{k}', 'method': f'{emb_name}+KMeans',
            'nmi': met_km['nmi'], 'ari': met_km['ari'],
            'n_communities_found': n_clusters
        })
        # HDBSCAN
        part_hdb = run_hdbscan(G_sub, emb, min_cluster_size=5, metric='euclidean', seed=42)
        if part_hdb:
            met_hdb = evaluate_communities(true_labels, part_hdb)
            results.append({
                'dataset': f'youtube_{k}', 'method': f'{emb_name}+HDBSCAN',
                'nmi': met_hdb['nmi'], 'ari': met_hdb['ari'],
                'n_communities_found': len(set(part_hdb.values()))
            })

    # DMoN
    part_dmon = run_dmon(G_sub, dimensions=64, epochs=200, lr=1e-3, seed=42, n_clusters=n_clusters)
    met_dmon = evaluate_communities(true_labels, part_dmon)
    results.append({
        'dataset': f'youtube_{k}', 'method': 'DMoN',
        'nmi': met_dmon['nmi'], 'ari': met_dmon['ari'],
        'n_communities_found': len(set(part_dmon.values()))
    })

df = pd.DataFrame(results)
df.to_csv('results/youtube_community_detection.csv', index=False)
df